# 2. Análise Básica da Rede

Este notebook executa a análise estrutural da nossa rede extraída (`edges_2days.csv`).
Vamos focar nas propriedades básicas de redes complexas como distribuição de graus, densidade, componentes conexas e métricas de distância.

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
import time
import os

# Opcional: configurar matplotlib para mostrar as figuras no notebook de forma mais elegante
%matplotlib inline

input_file = "../data/edges_2days.csv"
report_dir = "../report"
os.makedirs(report_dir, exist_ok=True)

print("1. Carregando os dados e construindo o grafo direcionado original...")
df = pd.read_csv(input_file, on_bad_lines='skip')
G_orig = nx.from_pandas_edgelist(df, source='source', target='target', create_using=nx.DiGraph())

num_v_orig = G_orig.number_of_nodes()
num_e_orig = G_orig.number_of_edges()
print(f"Grafo Original:\n  Vértices: {num_v_orig}\n  Arestas: {num_e_orig}")

### Simplificação do Grafo
Para a maioria das métricas clássicas (como diâmetro, componentes), usamos a versão não-direcionada e sem auto-loops da rede.

In [ ]:
print("Convertendo para grafo não-direcionado e removendo auto-loops...")
G = G_orig.to_undirected()
G.remove_edges_from(nx.selfloop_edges(G))

num_v = G.number_of_nodes()
num_e = G.number_of_edges()
print(f"Grafo Estático Não-Direcionado (Sem auto-loops):\n  Vértices: {num_v}\n  Arestas: {num_e}")

degrees = [d for n, d in G.degree()]
print(f"  Grau Mínimo: {np.min(degrees)}")
print(f"  Grau Máximo: {np.max(degrees)}")
print(f"  Grau Médio: {np.mean(degrees):.4f}")
print(f"  Densidade: {nx.density(G):.6f}")

### Maior Componente Conexa (LCC)
Em redes reais desconectadas, aplicamos as métricas mais pesadas apenas na maior parte conectada da rede.

In [ ]:
connected_components = list(nx.connected_components(G))
num_cc = len(connected_components)
print(f"Número de componentes conexas isoladas: {num_cc}")
cc_sizes = sorted([len(c) for c in connected_components], reverse=True)
print(f"Tamanho das 10 maiores componentes: {cc_sizes[:10]}")
if len(cc_sizes) > 10:
    print(f"O restante das {len(cc_sizes)-10} componentes variam de {min(cc_sizes[10:])} a {max(cc_sizes[10:])} nós.")

lcc_nodes = max(connected_components, key=len)
LCC = G.subgraph(lcc_nodes).copy()

print(f"\nMaior Componente Conexa (LCC):")
print(f"  Vértices: {LCC.number_of_nodes()} ({(LCC.number_of_nodes()/num_v)*100:.2f}% do total)")
print(f"  Arestas: {LCC.number_of_edges()} ({(LCC.number_of_edges()/num_e)*100:.2f}% do total)")

t0 = time.time()
avg_clustering = nx.average_clustering(LCC)
triangles = sum(nx.triangles(LCC).values()) // 3
print(f"\n  Coeficiente de clusterização médio: {avg_clustering:.4f}")
print(f"  Número de triângulos: {triangles}")
print(f"  (Tempo: {time.time()-t0:.2f}s)")

### Distâncias (Métricas Críticas)
Cálculos rigorosos de diâmetro têm complexidade O(V*E). Para redes com milhares de nós, usamos uma amostragem aleatória de pontos de partida.

In [ ]:
if LCC.number_of_nodes() > 5000:
    print("A LCC tem mais de 5000 nós. O cálculo exato pode demorar horas.")
    print("Aplicando amostragem em 500 nós escolhidos aleatoriamente para estimar as distâncias...")
    
    sampled_nodes = random.sample(list(LCC.nodes()), min(500, LCC.number_of_nodes()))
    path_lengths = []
    eccentricities = []
    
    for n in sampled_nodes:
        lengths = nx.single_source_shortest_path_length(LCC, n)
        if len(lengths) > 1:
            path_lengths.extend(list(lengths.values())[1:])
            eccentricities.append(max(lengths.values()))
    
    print(f"  Caminho Médio (Estimado): {np.mean(path_lengths):.4f}")
    print(f"  Diâmetro (Estimado): {max(eccentricities)}")
    print(f"  Raio (Estimado): {min(eccentricities)}")
else:
    t0 = time.time()
    print(f"  Caminho Médio: {nx.average_shortest_path_length(LCC):.4f}")
    print(f"  Diâmetro: {nx.diameter(LCC)}")
    print(f"  Raio: {nx.radius(LCC)}")
    print(f"  (Tempo: {time.time()-t0:.2f}s)")

### Visualização e Plots

In [ ]:
plt.figure(figsize=(8, 5))
degree_counts = nx.degree_histogram(G)
x = range(len(degree_counts))
y = [z / float(sum(degree_counts)) for z in degree_counts]

plt.loglog(x, y, color='b', marker='.')
plt.title("Distribuição de Graus (Grafo Total)")
plt.xlabel("Grau (k)")
plt.ylabel("P(k)")
plt.grid(True, which="both", ls="--")
plt.savefig(os.path.join(report_dir, "distribuicao_graus.png"))
plt.show()

In [ ]:
print("Gerando visualização reduzida para evitar sobrecarga na tela...")
core_graph = nx.k_core(LCC, k=min(np.max(degrees) // 2, 5)) 

if core_graph.number_of_nodes() < 10 or core_graph.number_of_nodes() > 1000:
    # Fallback para amostra ego graph do nó de maior grau
    max_node = max(dict(LCC.degree()).items(), key=lambda x: x[1])[0]
    core_graph = nx.ego_graph(LCC, max_node, radius=2)

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(core_graph, seed=42)
nx.draw(core_graph, pos, node_size=15, node_color='r', edge_color='gray', with_labels=False, alpha=0.7)
plt.title(f"Amostra da Rede (Sub-rede do LCC com {core_graph.number_of_nodes()} nós)")
plt.savefig(os.path.join(report_dir, "visualizacao_grafo.png"))
plt.show()